# Instruction-only compact handoff, plan4_dev, Kaggle T4

Same Tesla T4. Do not switch GPUs. Same Qwen3-4B revision, FP16, greedy decoding.

`plan4_dev` only. Do **not** load `plan4_hold`. Do not resume kvresult, kvswap, boundary, recon, iface, support, or hold. Do not regenerate A. Do not train an adapter.

R is the current fixed-prefix transferred baseline: reconstructed five-token prefix plus A's result-line keys and values.

**Cell 4** (`--phase construct`) freezes the first eligibility-instruction paragraph and three equal-size random selections. If `all_ok` is false, stop.

**Cell 5** (`--phase compare`) runs seven receiver conditions, 630 evaluations:

1. **r** — R alone.
2. **instr_kv** — instruction KV + R.
3. **r_instr_text** — R, with that paragraph in B's prompt as text.
4–6. **rand_1 / rand_2 / rand_3** — equal-size random KV + R.
7. **reconstruct** — reconstructed R.

Equal-size extra conditions keep the extra block after the shared prefix, then one newline, then R. Tokens, framing, and decoding stay unchanged. This is not the calculation-block comparison.

Download `/kaggle/working/results_stage_plan4_instr`.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan4_instr.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan4_instr.py").is_file() and (cand / "data" / "plan4_dev.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan4_instr.py and data/plan4_dev.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input"):
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    elif not (dest / "src" / "stage_plan4_instr.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
art = REPO / "results_stage_plan4_iface" / "a_artifacts.jsonl"
print("frozen A artifacts", art.is_file(), art)

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_instr")
print("Run construct first. Only compare if all_ok is true.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_instr", "--split", "plan4_dev", "--phase", "construct"])
print("plan4 instr construct exit", rc)
path = WORK / "results_stage_plan4_instr" / "construction.json"
if not path.is_file():
    path = REPO / "results" / "stage_plan4_instr" / "run" / "construction.json"
if path.is_file():
    data = json.loads(path.read_text(encoding="utf-8"))
    slim = {k: data.get(k) for k in ("n", "n_ok", "all_ok", "mean_n_instruction", "instruction_n_tokens_constant", "mean_random_overlap_instruction", "mean_random_overlap_calc", "frozen_rule")}
    print(json.dumps(slim, indent=2)[:16000])
    print(json.dumps(data.get("inspect"), indent=2)[:8000])

In [ ]:
src = WORK / "results_stage_plan4_instr" / "construction.json"
if not src.is_file():
    src = REPO / "results" / "stage_plan4_instr" / "run" / "construction.json"
check = json.loads(src.read_text(encoding="utf-8"))
if not check.get("all_ok"):
    raise RuntimeError("Construction failed. Do not run compare.")
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_instr", "--split", "plan4_dev", "--phase", "compare"])
print("plan4 instr compare exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_instr"
if not (src / "summary.json").is_file() and not (src / "construction.json").is_file():
    src = REPO / "results" / "stage_plan4_instr" / "run"
print("reading", src)
for name in ("construction.json", "summary.json"):
    path = src / name
    if path.is_file():
        data = json.loads(path.read_text(encoding="utf-8"))
        print("\n====", name, "====")
        keep = {k: data.get(k) for k in ("n", "n_ok", "all_ok", "n_rows", "mean_n_instruction", "receiver", "family_clustered_eligibility", "lead") if k in data or data.get(k) is not None}
        if "receiver" in keep and isinstance(keep["receiver"], dict):
            keep["receiver"] = {
                cond: {kk: (keep["receiver"].get(cond) or {}).get(kk) for kk in ("n", "parse_ok", "truncated", "plan_all_status_correct", "correct", "correct_with_wrong_eligibility", "mean_n_kept", "mean_n_extra", "mean_message_bytes", "mean_process_time_s")}
                for cond in keep["receiver"]
            }
        print(json.dumps(keep, indent=2)[:24000])